# Training BPE and Unigram Tokenizers

This notebook contains minimal implementations of the two most widely used subword tokenization algorithms:

- **Byte-Pair Encoding (BPE)** (Gage, 1994; Sennrich et al., 2016)
- **Unigram language model tokenization** (Kudo, 2018)

The code is written for readability, not speed, and uses only the Python standard library. The notebook has three parts:

1. Implementations of both algorithms
2. Training both tokenizers on a small corpus
3. A qualitative comparison of the resulting tokenizers

In [1]:
import math
import random
import re
import urllib.request
from collections import Counter

## Part 1: The algorithms

### 1.0 Pre-tokenization

Both algorithms operate on *words*: the text is first split into words and punctuation marks, and tokens never cross these boundaries. Training then only depends on how often each word occurs, so we can work with a table of word frequencies instead of the full corpus.

Following SentencePiece, a preceding space is kept as part of the word and written as `▁`. This marks word beginnings and makes it possible to recover the original text from the tokens. Both tokenizers use the same pre-tokenization, so all differences we observe later come from the algorithms themselves.

In [2]:
def pretokenize(text):
    # Split text into words and punctuation marks. A preceding space is kept as '▁'.
    text = " " + " ".join(text.split())  # normalize whitespace; the text start counts as a space
    return [word.replace(" ", "▁") for word in re.findall(r" ?\w+| ?[^\w\s]", text)]


def detokenize(tokens):
    return "".join(tokens).replace("▁", " ").strip()


words = pretokenize("Hello, world! It's 5 o'clock.")
print(words)
print(detokenize(words))

['▁Hello', ',', '▁world', '!', '▁It', "'", 's', '▁5', '▁o', "'", 'clock', '.']
Hello, world! It's 5 o'clock.


### 1.1 Byte-Pair Encoding (BPE)

**Training**

1. Split every word into characters. The initial vocabulary is the set of characters.
2. Count all pairs of adjacent symbols, weighted by word frequency.
3. Merge the most frequent pair into a new symbol, add it to the vocabulary, and record the merge.
4. Repeat steps 2–3 until the vocabulary has the desired size.

**Tokenization** of a new word: split it into characters and apply the learned merges in the order in which they were learned.

We start from characters here. Byte-level BPE, as used in GPT-2 and many later models, starts from the 256 possible bytes instead, so that no input is ever out of vocabulary.

In [3]:
def count_pairs(splits, word_freqs):
    # Count how often each pair of adjacent symbols occurs, weighted by word frequency.
    pair_freqs = Counter()
    for word, freq in word_freqs.items():
        symbols = splits[word]
        for pair in zip(symbols, symbols[1:]):
            pair_freqs[pair] += freq
    return pair_freqs


def merge_pair(symbols, pair):
    # Replace each occurrence of `pair` in a list of symbols by the merged symbol.
    merged = []
    i = 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
            merged.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    return merged

In [4]:
def train_bpe(word_freqs, vocab_size):
    # Learn merges from a dictionary {word: frequency}.
    # Returns the merges (in the order they were learned) and the vocabulary.

    # Start with each word split into characters; the initial vocabulary is the set of characters.
    splits = {word: list(word) for word in word_freqs}
    vocab = sorted(set("".join(word_freqs)))
    merges = []

    while len(vocab) < vocab_size:
        pair_freqs = count_pairs(splits, word_freqs)
        if not pair_freqs:
            break  # every word is a single symbol; nothing left to merge
        best_pair = max(pair_freqs, key=pair_freqs.get)
        new_symbol = best_pair[0] + best_pair[1]
        merges.append(best_pair)
        vocab.append(new_symbol)

        # Apply the merge to all words (only words containing the new symbol can change).
        for word in word_freqs:
            if new_symbol in word:
                splits[word] = merge_pair(splits[word], best_pair)

        if len(merges) % 500 == 0:
            print(f"{len(merges):5d} merges, vocabulary size {len(vocab)}")

    return merges, vocab

In [5]:
def bpe_tokenize_word(word, merge_rank):
    # Applying the merges in training order is equivalent to repeatedly applying
    # the applicable merge that was learned earliest, which is what we do here.
    symbols = list(word)
    while len(symbols) > 1:
        pairs = zip(symbols, symbols[1:])
        best_pair = min(pairs, key=lambda pair: merge_rank.get(pair, math.inf))
        if best_pair not in merge_rank:
            break  # no learned merge applies
        symbols = merge_pair(symbols, best_pair)
    return symbols


def bpe_tokenize(text, merges):
    merge_rank = {pair: rank for rank, pair in enumerate(merges)}
    return [token for word in pretokenize(text) for token in bpe_tokenize_word(word, merge_rank)]

The toy example from Sennrich et al. (2016), with `▁` marking word beginnings instead of an end-of-word symbol:

In [6]:
toy_word_freqs = Counter({"▁low": 5, "▁lower": 2, "▁newest": 6, "▁widest": 3})
toy_merges, toy_vocab = train_bpe(toy_word_freqs, vocab_size=20)

for i, (a, b) in enumerate(toy_merges, start=1):
    print(f"merge {i}: {a} + {b} -> {a + b}")

print(bpe_tokenize("lowest newer wider", toy_merges))

merge 1: e + s -> es
merge 2: es + t -> est
merge 3: ▁ + l -> ▁l
merge 4: ▁l + o -> ▁lo
merge 5: ▁lo + w -> ▁low
merge 6: ▁ + n -> ▁n
merge 7: ▁n + e -> ▁ne
merge 8: ▁ne + w -> ▁new
merge 9: ▁new + est -> ▁newest
['▁low', 'est', '▁new', 'e', 'r', '▁', 'w', 'i', 'd', 'e', 'r']


### 1.2 Unigram

**Model.** A Unigram tokenizer assigns a probability $p(t)$ to every token $t$ in the vocabulary. The probability of a segmentation $\mathbf{x} = (t_1, \dots, t_n)$ of a word is $P(\mathbf{x}) = \prod_i p(t_i)$. A word is tokenized into its most probable segmentation, which is found with the Viterbi algorithm.

**Training** starts from a large vocabulary and shrinks it:

1. Seed vocabulary: all characters plus the most frequent substrings of the words.
2. Estimate the token probabilities with EM. The E-step computes how often each token is expected to be used, summed over *all* possible segmentations of each word (with the forward-backward algorithm); the M-step normalizes these expected counts.
3. For each token, estimate how much the corpus likelihood would decrease if the token were removed. Remove the tokens with the smallest loss (here 25% of the vocabulary per round). Single characters are always kept, so every word can still be segmented.
4. Repeat steps 2–3 until the vocabulary has the desired size.

Compared to SentencePiece, this implementation makes a few simplifications: the seed vocabulary is found by brute-force substring counting, the loss in step 3 is computed from the best (Viterbi) segmentations only, and the M-step uses a small pseudo-count instead of SentencePiece's sparse prior.

In [7]:
MAX_TOKEN_LEN = 16    # longest token considered
UNK_LOG_PROB = -30.0  # log-probability assigned to characters not seen in training
PSEUDO_COUNT = 0.01   # small smoothing count, so that no token gets probability zero


def seed_vocab(word_freqs, seed_size):
    # All characters plus the most frequent longer substrings.
    # Returns {token: probability}, with probabilities proportional to substring frequency.
    substring_freqs = Counter()
    for word, freq in word_freqs.items():
        for start in range(len(word)):
            for end in range(start + 1, min(start + MAX_TOKEN_LEN, len(word)) + 1):
                substring_freqs[word[start:end]] += freq
    chars = [s for s in substring_freqs if len(s) == 1]
    longer = [s for s, _ in substring_freqs.most_common() if len(s) > 1]
    seed = chars + longer[: seed_size - len(chars)]
    total = sum(substring_freqs[s] for s in seed)
    return {s: substring_freqs[s] / total for s in seed}

In [8]:
# Words are short, so for the forward-backward computation we can use plain
# probabilities instead of log-probabilities without running into underflow.

def forward(word, probs):
    # alpha[i] = total probability of all segmentations of word[:i]
    alpha = [1.0] + [0.0] * len(word)
    for end in range(1, len(word) + 1):
        for start in range(max(0, end - MAX_TOKEN_LEN), end):
            token = word[start:end]
            if token in probs:
                alpha[end] += alpha[start] * probs[token]
    return alpha


def backward(word, probs):
    # beta[i] = total probability of all segmentations of word[i:]
    n = len(word)
    beta = [0.0] * n + [1.0]
    for start in range(n - 1, -1, -1):
        for end in range(start + 1, min(start + MAX_TOKEN_LEN, n) + 1):
            token = word[start:end]
            if token in probs:
                beta[start] += probs[token] * beta[end]
    return beta


def em_step(word_freqs, probs):
    # One EM iteration. Returns the new probabilities and the corpus log-likelihood.
    expected_counts = Counter()
    log_likelihood = 0.0
    for word, freq in word_freqs.items():
        alpha, beta = forward(word, probs), backward(word, probs)
        word_prob = alpha[-1]  # probability of the word, summed over all segmentations
        log_likelihood += freq * math.log(word_prob)
        # E-step: posterior probability that word[start:end] is one token of the segmentation
        for start in range(len(word)):
            for end in range(start + 1, min(start + MAX_TOKEN_LEN, len(word)) + 1):
                token = word[start:end]
                if token in probs:
                    posterior = alpha[start] * probs[token] * beta[end] / word_prob
                    expected_counts[token] += freq * posterior
    # M-step: the new probabilities are the normalized expected counts
    total = sum(expected_counts[token] + PSEUDO_COUNT for token in probs)
    new_probs = {token: (expected_counts[token] + PSEUDO_COUNT) / total for token in probs}
    return new_probs, log_likelihood

In [9]:
def viterbi(word, probs, exclude=None):
    # Most probable segmentation of a word. Returns (tokens, log-probability).
    # `exclude` lets us pretend that one token is not in the vocabulary (needed for pruning).
    n = len(word)
    best_score = [0.0] + [-math.inf] * n  # log-prob of the best segmentation of word[:i]
    best_start = [0] * (n + 1)            # start position of the last token in that segmentation
    for end in range(1, n + 1):
        for start in range(max(0, end - MAX_TOKEN_LEN), end):
            token = word[start:end]
            if token in probs and token != exclude:
                score = best_score[start] + math.log(probs[token])
            elif len(token) == 1:
                score = best_score[start] + UNK_LOG_PROB  # unknown character
            else:
                continue
            if score > best_score[end]:
                best_score[end], best_start[end] = score, start
    # Follow the back-pointers to recover the tokens.
    tokens, end = [], n
    while end > 0:
        tokens.insert(0, word[best_start[end]:end])
        end = best_start[end]
    return tokens, best_score[n]


def unigram_tokenize(text, probs):
    return [token for word in pretokenize(text) for token in viterbi(word, probs)[0]]

In [10]:
def prune(word_freqs, probs, n_remove):
    # Remove the n_remove tokens whose removal decreases the likelihood the least.
    # The loss of a token is estimated from the best segmentations: for every word whose
    # best segmentation uses the token, how much worse is the best segmentation without it?
    loss = {token: 0.0 for token in probs}
    for word, freq in word_freqs.items():
        tokens, score = viterbi(word, probs)
        for token in set(tokens):
            if len(token) > 1:
                _, score_without = viterbi(word, probs, exclude=token)
                loss[token] += freq * (score - score_without)
    candidates = sorted((t for t in probs if len(t) > 1), key=lambda t: loss[t])  # never single characters
    to_remove = set(candidates[:n_remove])
    return {t: p for t, p in probs.items() if t not in to_remove}


def train_unigram(word_freqs, vocab_size, seed_size=None, shrink_factor=0.75, em_iterations=2):
    # Train a Unigram tokenizer on {word: frequency}. Returns {token: probability}.
    assert vocab_size >= len(set("".join(word_freqs))), "vocab_size must cover all characters"
    probs = seed_vocab(word_freqs, seed_size or 10 * vocab_size)
    while True:
        for _ in range(em_iterations):
            probs, log_likelihood = em_step(word_freqs, probs)
        print(f"vocabulary size {len(probs):6d}, log-likelihood {log_likelihood:,.0f}")
        if len(probs) <= vocab_size:
            return probs
        # Keep shrink_factor of the vocabulary, but do not go below vocab_size.
        n_remove = min(len(probs) - vocab_size, int(len(probs) * (1 - shrink_factor)))
        probs = prune(word_freqs, probs, n_remove)

## Part 2: Training on a small corpus

We use *Tiny Shakespeare* (about 1 MB of text from Shakespeare's plays), a corpus commonly used for small-scale experiments. Any other plain-text corpus can be used by changing `CORPUS_URL` or by assigning a string to `corpus` directly.

In [11]:
CORPUS_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
corpus = urllib.request.urlopen(CORPUS_URL).read().decode("utf-8")

print(f"{len(corpus):,} characters\n")
print(corpus[:250])

1,115,394 characters

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [12]:
word_freqs = Counter(pretokenize(corpus))
print(f"{sum(word_freqs.values()):,} pre-tokens, {len(word_freqs):,} distinct")
print(word_freqs.most_common(20))

262,927 pre-tokens, 14,006 distinct
[(',', 19846), (':', 10314), ('.', 7885), ('▁the', 5438), ("'", 5306), ('▁I', 5019), ('▁to', 4107), ('▁and', 3756), (';', 3628), ('▁of', 3313), ('▁you', 2855), ('▁my', 2677), ('▁a', 2629), ('?', 2462), ('!', 2172), ('▁in', 2151), ('▁that', 1986), ('▁And', 1927), ('▁is', 1906), ('▁not', 1899)]


Both tokenizers are trained to the same vocabulary size. With this implementation, training BPE takes one to two minutes; reduce `VOCAB_SIZE` for a quicker run.

In [13]:
VOCAB_SIZE = 3000

In [14]:
bpe_merges, bpe_vocab = train_bpe(word_freqs, VOCAB_SIZE)

print("\nFirst merges:", [a + b for a, b in bpe_merges[:20]])
print("Last merges: ", [a + b for a, b in bpe_merges[-20:]])

  500 merges, vocabulary size 564


 1000 merges, vocabulary size 1064


 1500 merges, vocabulary size 1564


 2000 merges, vocabulary size 2064


 2500 merges, vocabulary size 2564



First merges: ['▁t', 'he', '▁a', 'ou', '▁s', '▁m', '▁w', 'in', 're', 'ha', '▁the', 'nd', '▁b', 'is', 'or', '▁f', '▁I', 'er', 'll', 'it']
Last merges:  ['▁early', 'LEY', '▁cred', '▁EX', '▁Mercut', '▁Mercutio', '▁Lew', '▁Lewis', '▁ANTI', '▁ANTIG', '▁ANTIGON', '▁ANTIGONUS', '▁hung', 'ating', 'orthy', 'under', '▁Alack', '▁idle', '▁minister', 'house']


In [15]:
unigram_probs = train_unigram(word_freqs, VOCAB_SIZE)

by_prob = sorted(unigram_probs, key=unigram_probs.get, reverse=True)
print("\nMost probable tokens:", by_prob[:20])

vocabulary size  30000, log-likelihood -1,878,427


vocabulary size  22500, log-likelihood -1,834,262


vocabulary size  16875, log-likelihood -1,833,503


vocabulary size  12657, log-likelihood -1,833,367


vocabulary size   9493, log-likelihood -1,833,319


vocabulary size   7120, log-likelihood -1,835,278


vocabulary size   5340, log-likelihood -1,850,154


vocabulary size   4005, log-likelihood -1,878,002


vocabulary size   3004, log-likelihood -1,923,138


vocabulary size   3000, log-likelihood -1,923,282

Most probable tokens: [',', ':', 's', '.', '▁the', '▁I', "'", '▁to', '▁and', ';', 'd', '▁of', '▁a', '▁you', '▁my', '?', '▁in', '!', '▁that', '▁me']


## Part 3: Qualitative comparison

### 3.1 Sentences

Tokens are separated by spaces; `▁` marks the beginning of a word.

In [16]:
def compare(text):
    print("BPE:    ", " ".join(bpe_tokenize(text, bpe_merges)))
    print("Unigram:", " ".join(unigram_tokenize(text, unigram_probs)))
    print()


compare("Now is the winter of our discontent made glorious summer by this sun of York.")
compare("To be, or not to be, that is the question.")
compare("Tokenizers are trained on large corpora before the language model itself.")

BPE:     ▁Now ▁is ▁the ▁win ter ▁of ▁our ▁disc ontent ▁made ▁gl orious ▁summer ▁by ▁this ▁sun ▁of ▁York .
Unigram: ▁Now ▁is ▁the ▁winter ▁of ▁our ▁discontent ▁made ▁glorious ▁summer ▁by ▁this ▁sun ▁of ▁York .

BPE:     ▁To ▁be , ▁or ▁not ▁to ▁be , ▁that ▁is ▁the ▁quest ion .
Unigram: ▁To ▁be , ▁or ▁not ▁to ▁be , ▁that ▁is ▁the ▁question .

BPE:     ▁To ken iz ers ▁are ▁tra ined ▁on ▁l ar ge ▁cor p or a ▁before ▁the ▁lan gu age ▁mod el ▁itself .
Unigram: ▁To ken ize r s ▁are ▁tra in ed ▁on ▁large ▁co r p or a ▁before ▁the ▁language ▁mo d el ▁itself .



### 3.2 Words

Words with derivational or inflectional morphology, followed by a few words that are unlikely to occur in the training corpus. Rows where the two tokenizers differ are marked with `*`.

In [17]:
words = [
    "unkindness", "unbearable", "forgiveness", "remembered", "daughters", "undone",
    "uncrowned", "happiest", "mightiest", "brotherhood",
    "tokenization", "Shakespearean", "smartphone", "linguistics",
]

print(f"{'word':16}{'BPE':30}{'Unigram':30}")
for word in words:
    bpe_tokens = bpe_tokenize(word, bpe_merges)
    unigram_tokens = unigram_tokenize(word, unigram_probs)
    marker = "*" if bpe_tokens != unigram_tokens else ""
    print(f"{word:16}{' '.join(bpe_tokens):30}{' '.join(unigram_tokens):30}{marker}")

word            BPE                           Unigram                       
unkindness      ▁unk ind ness                 ▁unkind ness                  *
unbearable      ▁un b ear able                ▁un bear able                 *
forgiveness     ▁forg i ven ess               ▁forgive ness                 *
remembered      ▁remem be red                 ▁remember ed                  *
daughters       ▁d aught ers                  ▁daughter s                   *
undone          ▁und one                      ▁undone                       *
uncrowned       ▁unc rown ed                  ▁un c row n ed                *
happiest        ▁happ iest                    ▁hap p ies t                  *
mightiest       ▁might iest                   ▁might ies t                  *
brotherhood     ▁brother hood                 ▁brother hood                 
tokenization    ▁to ken iz ation              ▁to ken i z ation             *
Shakespearean   ▁Sha kes p ea re an           ▁S h ake s pe ar e a

Some observations with the default settings:

- For words whose parts are frequent in the corpus, Unigram tends to split at morpheme boundaries (*unkind|ness*, *forgive|ness*, *remember|ed*), whereas BPE segments reflect the order in which merges were learned (*unk|ind|ness*, *forg|i|ven|ess*). This is in line with the findings of Bostrom and Durrett (2020).
- Unigram represents more complete words as single tokens (*winter*, *question*, *language*).
- For words far from the training data, Unigram falls back to very short pieces and single characters more readily than BPE (*Shakespearean*, *smartphone*), and BPE's segmentation is sometimes the more plausible one (*happ|iest* vs. *hap|p|ies|t*).

### 3.3 Vocabularies

Both vocabularies have the same size. How much do they overlap, and what do the tokens that only one of them contains look like?

In [18]:
bpe_set, unigram_set = set(bpe_vocab), set(unigram_probs)
print(f"Shared tokens: {len(bpe_set & unigram_set)} of {VOCAB_SIZE}")
print(f"Average token length: BPE {sum(map(len, bpe_set)) / len(bpe_set):.2f}, "
      f"Unigram {sum(map(len, unigram_set)) / len(unigram_set):.2f}\n")

print("Longest tokens only in BPE:    ", sorted(bpe_set - unigram_set, key=len, reverse=True)[:15])
print("Longest tokens only in Unigram:", sorted(unigram_set - bpe_set, key=len, reverse=True)[:15])

Shared tokens: 1668 of 3000
Average token length: BPE 4.73, Unigram 5.90

Longest tokens only in BPE:     ['▁NORTHUMBER', '▁Northumber', '▁Plantagen', '▁BOLINGBRO', '▁GLOUCEST', '▁brothers', '▁CORIOLAN', '▁Citizens', '▁ELIZABET', '▁Bucking', '▁citizen', '▁BOLINGB', '▁honours', '▁Plantag', '▁ANTIGON']
Longest tokens only in Unigram: ['▁proclamation', '▁intelligence', '▁commonwealth', '▁imprisonment', '▁circumstance', '▁approbation', '▁adversaries', '▁countenance', '▁counterfeit', '▁gentlewoman', '▁Conspirator', '▁treacherous', '▁remembrance', '▁disposition', '▁substitute']


### 3.4 Which vocabulary items are actually used?

We tokenize the training corpus with both tokenizers and count how often each vocabulary item occurs. In BPE, a token created by a merge may later be absorbed into longer tokens almost everywhere it occurs. It still occupies a slot in the vocabulary, but is rarely or never produced.

In [19]:
bpe_usage = Counter(bpe_tokenize(corpus, bpe_merges))
unigram_usage = Counter(unigram_tokenize(corpus, unigram_probs))

for name, vocab, usage in [("BPE", bpe_vocab, bpe_usage), ("Unigram", unigram_probs, unigram_usage)]:
    unused = [t for t in vocab if usage[t] == 0]
    print(f"{name:8} corpus length {sum(usage.values()):,} tokens; {len(unused)} vocabulary items never used")

bpe_unused = [t for t in bpe_vocab if bpe_usage[t] == 0]
print("\nExamples of unused BPE tokens:", sorted(bpe_unused, key=len, reverse=True)[:15])

BPE      corpus length 319,421 tokens; 232 vocabulary items never used
Unigram  corpus length 309,079 tokens; 2 vocabulary items never used

Examples of unused BPE tokens: ['▁NORTHUMBER', '▁Northumber', '▁BOLINGBRO', '▁GLOUCEST', '▁CORIOLAN', '▁ELIZABET', '▁PETRUCH', '▁ISABELL', '▁BUCKING', '▁BOLINGB', '▁BAPTIST', '▁AUTOLYC', '▁PROSPER', '▁Serving', '▁HASTING']


Most unused BPE tokens are prefixes of character names. Speaker labels such as *NORTHUMBERLAND* are frequent in the plays, and BPE builds them one merge at a time (compare the last merges printed in Part 2: *▁ANTI*, *▁ANTIG*, *▁ANTIGON*, *▁ANTIGONUS*). Once the full name exists, the intermediate tokens are no longer produced. Unigram's pruning step removes such tokens, since removing them barely affects the likelihood. The freed-up vocabulary slots go to other tokens, and the corpus is encoded with fewer tokens overall.

### 3.5 Unigram is a probabilistic model

Unigram does not just provide one segmentation per word: it defines a probability distribution over all segmentations. For short words we can simply enumerate them.

In [20]:
def all_segmentations(word, probs):
    # All segmentations of `word` into vocabulary tokens, with their probabilities.
    if not word:
        return [([], 1.0)]
    results = []
    for end in range(1, min(len(word), MAX_TOKEN_LEN) + 1):
        token = word[:end]
        if token in probs:
            for rest, p in all_segmentations(word[end:], probs):
                results.append(([token] + rest, probs[token] * p))
    return results


word = "▁unbearable"
segmentations = sorted(all_segmentations(word, unigram_probs), key=lambda s: s[1], reverse=True)
total = sum(p for _, p in segmentations)

print(f"{len(segmentations)} segmentations of {word}; the most probable ones:")
for tokens, p in segmentations[:5]:
    print(f"  P = {p / total:.4f}   {' '.join(tokens)}")

78 segmentations of ▁unbearable; the most probable ones:
  P = 0.9950   ▁un bear able
  P = 0.0027   ▁un be ar able
  P = 0.0013   ▁un bear a ble
  P = 0.0008   ▁ un bear able
  P = 0.0001   ▁un b e ar able


This distribution can be used for *subword regularization* (Kudo, 2018): during training of a downstream model, each word is segmented by sampling from $P(\mathbf{x})^\alpha$ instead of always using the best segmentation. Smaller values of $\alpha$ give more diverse segmentations. (BPE-dropout (Provilkov et al., 2020) is an analogous technique for BPE.)

In [21]:
ALPHA = 0.2
random.seed(0)
weights = [p ** ALPHA for _, p in segmentations]
for _ in range(8):
    tokens, _ = random.choices(segmentations, weights=weights)[0]
    print(" ".join(tokens))

▁un b e a ra ble
▁un be a ra ble
▁un bear a ble
▁un bear able
▁ un bear able
▁un bear a ble
▁ un bear a ble
▁un bear able


## References

- Gage (1994). A New Algorithm for Data Compression. *C Users Journal*.
- Sennrich, Haddow, and Birch (2016). Neural Machine Translation of Rare Words with Subword Units. *ACL*.
- Kudo (2018). Subword Regularization: Improving Neural Network Translation Models with Multiple Subword Candidates. *ACL*.
- Kudo and Richardson (2018). SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing. *EMNLP (System Demonstrations)*.
- Bostrom and Durrett (2020). Byte Pair Encoding is Suboptimal for Language Model Pretraining. *Findings of EMNLP*.
- Provilkov, Emelianenko, and Voita (2020). BPE-Dropout: Simple and Effective Subword Regularization. *ACL*.